# Assignment 2: Zipf's Law and Tokenizer Analytics

This notebook studies whether tokens follow Zipf's law across Wikipedia text, languages, and model tokenizers. It compares English, Hindi, and Arabic; word-level tokenization; and Llama, Qwen, and Kimi-style Hugging Face tokenizers.

Main questions answered here:

- Do tokens follow Zipf's law?
- How do rank-frequency plots change across languages and tokenizers?
- How do vocabulary size, morphology, script, and byte/subword behavior affect token statistics?
- Where is the sweet spot to stop tokenizer training?
- Can we decide vocabulary size algorithmically from emerging patterns?
- For a language, how many tokens are needed to estimate a Zipf-supported tokenizer/training corpus?

The code downloads tokenizer files only, not model weights.

## 0. Setup

Run the install cell if your environment is missing packages. The rest of the notebook is written to fail gracefully: if a model tokenizer is gated or unavailable, the analysis continues with the tokenizers that load successfully.

In [ ]:
# Uncomment if needed.
# %pip install -q numpy pandas matplotlib requests transformers tokenizers sentencepiece tiktoken protobuf


In [ ]:
import math
import re
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)
warnings.filterwarnings("ignore")

## 1. Collect Wikipedia Text

The next cell fetches random main-namespace Wikipedia pages for English, Hindi, and Arabic. Increase `PAGES_PER_LANGUAGE` for a stronger experiment. For a class assignment, 20-50 pages per language is usually enough to see the Zipf curve; for stable estimates, use hundreds or thousands of pages.

In [ ]:
LANGUAGES = {
    "en": "English",
    "hi": "Hindi",
    "ar": "Arabic",
}

PAGES_PER_LANGUAGE = 30
CHARS_PER_PAGE = 5000

FALLBACK_TEXTS = {
    "en": [
        "Wikipedia is a multilingual free online encyclopedia written and maintained by a community of volunteers. Articles cover history, science, culture, technology, geography, literature, and many other areas of human knowledge. Natural language contains a small number of very common words and a long tail of rare words, names, and technical terms."
    ],
    "hi": [
        "विकिपीडिया एक बहुभाषी मुक्त ज्ञानकोश है जिसे स्वयंसेवकों का समुदाय लिखता और संपादित करता है। लेख इतिहास, विज्ञान, संस्कृति, प्रौद्योगिकी, भूगोल और साहित्य जैसे अनेक विषयों पर आधारित होते हैं। प्राकृतिक भाषा में कुछ शब्द बहुत बार आते हैं और बहुत से शब्द बहुत कम बार दिखाई देते हैं।"
    ],
    "ar": [
        "ويكيبيديا موسوعة حرة متعددة اللغات يكتبها ويحررها مجتمع من المتطوعين. تغطي المقالات التاريخ والعلوم والثقافة والتقنية والجغرافيا والأدب ومجالات كثيرة من المعرفة الإنسانية. في اللغة الطبيعية تظهر كلمات قليلة بتكرار عال بينما توجد ذيل طويل من الكلمات النادرة والأسماء والمصطلحات."
    ],
}

def clean_wiki_text(text):
    text = re.sub(r"\s+", " ", text or "").strip()
    return text

def fetch_random_wikipedia(lang, n_pages=30, chars_per_page=5000, pause=0.2):
    session = requests.Session()
    session.headers.update({
        "User-Agent": "ZipfTokenizerAssignment/1.0 (student research notebook; contact: local-notebook)"
    })
    url = f"https://{lang}.wikipedia.org/w/api.php"
    texts = []
    while len(texts) < n_pages:
        batch = min(10, n_pages - len(texts))
        params = {
            "action": "query",
            "generator": "random",
            "grnnamespace": 0,
            "grnlimit": batch,
            "prop": "extracts",
            "explaintext": 1,
            "format": "json",
            "formatversion": 2,
        }
        response = session.get(url, params=params, timeout=30)
        response.raise_for_status()
        pages = response.json().get("query", {}).get("pages", [])
        for page in pages:
            text = clean_wiki_text(page.get("extract", ""))[:chars_per_page]
            if len(text) > 200:
                texts.append(text)
        time.sleep(pause)
    return texts[:n_pages]

wiki_texts = {}
for lang, name in LANGUAGES.items():
    try:
        wiki_texts[lang] = fetch_random_wikipedia(lang, PAGES_PER_LANGUAGE, CHARS_PER_PAGE)
        print(f"{name}: fetched {len(wiki_texts[lang])} pages, {sum(len(t) for t in wiki_texts[lang]):,} characters")
    except Exception as exc:
        print(f"{name}: Wikipedia fetch failed ({type(exc).__name__}: {exc}). Using tiny fallback sample.")
        wiki_texts[lang] = FALLBACK_TEXTS[lang]


In [ ]:
corpus = {lang: "\n\n".join(texts) for lang, texts in wiki_texts.items()}
corpus_stats = pd.DataFrame([
    {
        "language": LANGUAGES[lang],
        "pages": len(texts),
        "characters": len(corpus[lang]),
        "bytes_utf8": len(corpus[lang].encode("utf-8")),
    }
    for lang, texts in wiki_texts.items()
])
display(corpus_stats)

## 2. Zipf Utilities

Zipf's law says that token frequency is approximately inversely proportional to rank:

$$ f(r) \propto r^{-\alpha} $$

For many natural-language word distributions, `alpha` is often near 1 over the middle rank range. Real corpora deviate at the head and tail because of stop words, names, formatting, finite sample size, and tokenizer design.

In [ ]:
WORD_RE = re.compile(r"\w+", flags=re.UNICODE)

def word_tokenize(text):
    return WORD_RE.findall(text.lower())

def fit_zipf(counter, min_rank=10, max_rank=5000):
    freqs = np.array(sorted(counter.values(), reverse=True), dtype=float)
    if len(freqs) < min_rank + 5:
        return {"alpha": np.nan, "r2": np.nan, "fit_start": np.nan, "fit_end": np.nan}
    ranks = np.arange(1, len(freqs) + 1, dtype=float)
    end = int(min(max_rank, len(freqs)))
    mask = (ranks >= min_rank) & (ranks <= end) & (freqs > 1)
    if mask.sum() < 5:
        mask = (ranks >= 1) & (ranks <= end) & (freqs > 0)
    x = np.log10(ranks[mask])
    y = np.log10(freqs[mask])
    slope, intercept = np.polyfit(x, y, 1)
    y_hat = slope * x + intercept
    ss_res = np.sum((y - y_hat) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    return {"alpha": -slope, "r2": r2, "fit_start": int(ranks[mask][0]), "fit_end": int(ranks[mask][-1])}

def frequency_frame(tokens, label, language, vocab_size_available=None):
    counter = Counter(tokens)
    rows = []
    for rank, (token, freq) in enumerate(counter.most_common(), start=1):
        rows.append({"rank": rank, "token": token, "frequency": freq})
    df = pd.DataFrame(rows)
    if len(df):
        df["relative_frequency"] = df["frequency"] / df["frequency"].sum()
        df["log_rank"] = np.log10(df["rank"])
        df["log_frequency"] = np.log10(df["frequency"])
    z = fit_zipf(counter)
    summary = {
        "language": language,
        "tokenizer": label,
        "tokens_total": len(tokens),
        "types_observed": len(counter),
        "vocab_size_available": vocab_size_available,
        "type_token_ratio": len(counter) / len(tokens) if tokens else np.nan,
        "singleton_ratio": sum(1 for v in counter.values() if v == 1) / len(counter) if counter else np.nan,
        **z,
    }
    return df, summary, counter

def plot_zipf(freq_frames, title, max_rank=20000):
    plt.figure(figsize=(9, 6))
    for label, df in freq_frames.items():
        if len(df) == 0:
            continue
        view = df[df["rank"] <= max_rank]
        plt.plot(view["rank"], view["frequency"], linewidth=1.8, label=label)
    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("Rank (log scale)")
    plt.ylabel("Frequency (log scale)")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

## 3. Word-Level Baseline: Do Wikipedia Words Follow Zipf's Law?

Before studying model tokenizers, first test ordinary word-like tokens. This gives a baseline for each language.

In [ ]:
all_freq_frames = {}
all_counters = {}
summaries = []

for lang, text in corpus.items():
    tokens = word_tokenize(text)
    df, summary, counter = frequency_frame(tokens, "word_regex", LANGUAGES[lang])
    all_freq_frames[(lang, "word_regex")] = df
    all_counters[(lang, "word_regex")] = counter
    summaries.append(summary)

word_summary = pd.DataFrame(summaries)
display(word_summary.sort_values(["language", "tokenizer"]))

for lang in LANGUAGES:
    plot_zipf({"word_regex": all_freq_frames[(lang, "word_regex")]}, f"Word-level Zipf plot: {LANGUAGES[lang]}")

**Interpretation guide:** if `alpha` is close to 1 and `r2` is high on the log-log fit, the token distribution approximately follows Zipf's law. Do not expect a perfect line: Wikipedia titles, numbers, names, morphology, and finite sample size bend the curve.

## 4. Load Model Tokenizers

The notebook tries these tokenizer repositories:

- Llama: `meta-llama/Meta-Llama-3.1-8B`, with a fallback test Llama tokenizer if the Meta repo is gated.
- Qwen: `Qwen/Qwen2.5-7B`.
- Kimi: `moonshotai/Kimi-K2-Instruct`.

Only tokenizers are loaded. No model weights are downloaded.

In [ ]:
TOKENIZER_SPECS = {
    "Llama": {
        "ids": ["meta-llama/Meta-Llama-3.1-8B", "meta-llama/Llama-3.1-8B", "hf-internal-testing/llama-tokenizer"],
        "trust_remote_code": False,
    },
    "Qwen2.5": {
        "ids": ["Qwen/Qwen2.5-7B"],
        "trust_remote_code": False,
    },
    "Kimi-K2": {
        "ids": ["moonshotai/Kimi-K2-Instruct"],
        "trust_remote_code": True,
    },
}

def load_tokenizers(specs):
    try:
        from transformers import AutoTokenizer
    except Exception as exc:
        print(f"transformers is not available: {exc}")
        return {}, []

    loaded = {}
    status_rows = []
    for label, spec in specs.items():
        last_error = None
        for model_id in spec["ids"]:
            try:
                tok = AutoTokenizer.from_pretrained(
                    model_id,
                    use_fast=True,
                    trust_remote_code=spec.get("trust_remote_code", False),
                )
                loaded[label] = tok
                status_rows.append({
                    "tokenizer": label,
                    "loaded_from": model_id,
                    "vocab_size_attribute": getattr(tok, "vocab_size", None),
                    "len_tokenizer": len(tok),
                    "status": "loaded",
                })
                break
            except Exception as exc:
                last_error = exc
        if label not in loaded:
            status_rows.append({
                "tokenizer": label,
                "loaded_from": None,
                "vocab_size_attribute": None,
                "len_tokenizer": None,
                "status": f"failed: {type(last_error).__name__}: {last_error}",
            })
    return loaded, status_rows

model_tokenizers, tokenizer_status = load_tokenizers(TOKENIZER_SPECS)
display(pd.DataFrame(tokenizer_status))

## 5. Tokenizer Comparison Across Languages

For each language and tokenizer, this computes:

- total tokens produced,
- observed token types,
- tokenizer vocabulary size,
- characters per token and bytes per token,
- Zipf exponent `alpha`,
- fit quality `r2`,
- singleton ratio in the observed corpus.

In [ ]:
def encode_ids(tokenizer, text):
    return tokenizer.encode(text, add_special_tokens=False)

tokenizer_summaries = []

for lang, text in corpus.items():
    for label, tok in model_tokenizers.items():
        ids = encode_ids(tok, text)
        token_keys = [str(i) for i in ids]
        df, summary, counter = frequency_frame(
            token_keys,
            label,
            LANGUAGES[lang],
            vocab_size_available=len(tok),
        )
        summary["chars_per_token"] = len(text) / len(ids) if ids else np.nan
        summary["bytes_per_token"] = len(text.encode("utf-8")) / len(ids) if ids else np.nan
        summary["loaded_vocab"] = len(tok)
        all_freq_frames[(lang, label)] = df
        all_counters[(lang, label)] = counter
        tokenizer_summaries.append(summary)

comparison = pd.DataFrame(summaries + tokenizer_summaries)
display(comparison.sort_values(["language", "tokenizer"]))

In [ ]:
for lang in LANGUAGES:
    frames = {
        label: df
        for (l, label), df in all_freq_frames.items()
        if l == lang
    }
    plot_zipf(frames, f"Zipf plot by tokenizer: {LANGUAGES[lang]}")

if not comparison.empty:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    plot_df = comparison[comparison["tokenizer"] != "word_regex"].copy()
    if len(plot_df):
        for tok_label, g in plot_df.groupby("tokenizer"):
            axes[0].plot(g["language"], g["chars_per_token"], marker="o", label=tok_label)
            axes[1].plot(g["language"], g["alpha"], marker="o", label=tok_label)
        axes[0].set_title("Compression: characters per token")
        axes[0].set_ylabel("Characters per token")
        axes[1].set_title("Zipf alpha by tokenizer")
        axes[1].set_ylabel("alpha")
        axes[1].axhline(1.0, color="black", linestyle="--", linewidth=1, alpha=0.6)
        for ax in axes:
            ax.tick_params(axis="x", rotation=25)
            ax.legend()
    plt.tight_layout()
    plt.show()

## 6. What Changes Across English, Hindi, and Arabic?

Use the table and plots above to make the following observations:

- **English** often compresses well in tokenizers trained heavily on English and code. More common words become whole tokens.
- **Hindi** uses Devanagari and has different morphology and word formation. If the tokenizer has weaker Hindi coverage, words split into more subword/byte pieces, lowering `chars_per_token` and increasing token count.
- **Arabic** has rich morphology, clitics, and a right-to-left script. Tokenizers may split prefixes/suffixes differently, changing both the Zipf curve and the singleton tail.
- **Vocabulary size matters**, but not alone. A larger vocabulary helps only if its merges/tokens were trained on the language distribution you care about.
- **Tokenizer design matters.** Byte-level BPE, SentencePiece unigram/BPE, and custom multilingual tokenizers have different behavior on whitespace, Unicode bytes, unknown text, and rare scripts.

A useful practical metric is `chars_per_token` or `bytes_per_token`: if one language needs many more tokens for the same text, the LM spends more context length and training compute representing that language.

## 7. Sweet Spot for Tokenizer Training

A tokenizer vocabulary should be large enough to compress frequent patterns but not so large that it memorizes rare words and wastes embedding parameters. The experiment below trains byte-level BPE tokenizers on this Wikipedia sample at several vocabulary sizes and measures whether extra vocabulary still improves compression.

A reasonable stopping rule:

1. Train tokenizers at increasing vocabulary sizes.
2. Measure tokens per character or bytes per token on held-out text.
3. Stop when marginal compression improvement falls below a threshold, for example 1 percent, for two consecutive increases.
4. Also check the tail: if many observed token types are singletons, the tokenizer may be overfitting rare strings.
5. For multilingual tokenizers, check each language separately instead of optimizing only the average.

In [ ]:
VOCAB_SIZES_TO_TRY = [1000, 2000, 4000, 8000, 16000, 32000]

def train_byte_bpe(texts, vocab_size):
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
    tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tokenizer.decoder = decoders.ByteLevel()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=2,
        special_tokens=["[UNK]"],
    )
    tokenizer.train_from_iterator(texts, trainer=trainer)
    return tokenizer

def evaluate_trained_tokenizer(tokenizer, texts_by_lang):
    rows = []
    for lang, text in texts_by_lang.items():
        ids = tokenizer.encode(text).ids
        counter = Counter(ids)
        z = fit_zipf(counter)
        rows.append({
            "language": LANGUAGES[lang],
            "tokens_total": len(ids),
            "types_observed": len(counter),
            "chars_per_token": len(text) / len(ids) if ids else np.nan,
            "bytes_per_token": len(text.encode("utf-8")) / len(ids) if ids else np.nan,
            "singleton_ratio": sum(1 for v in counter.values() if v == 1) / len(counter) if counter else np.nan,
            **z,
        })
    return rows

trained_rows = []
try:
    train_texts = []
    for lang, texts in wiki_texts.items():
        train_texts.extend(texts)
    for vocab_size in VOCAB_SIZES_TO_TRY:
        bpe = train_byte_bpe(train_texts, vocab_size)
        for row in evaluate_trained_tokenizer(bpe, corpus):
            row["vocab_size"] = vocab_size
            trained_rows.append(row)
        print(f"trained vocab_size={vocab_size}")
    trained_eval = pd.DataFrame(trained_rows)
    display(trained_eval)
except Exception as exc:
    print(f"Tokenizer training experiment skipped: {type(exc).__name__}: {exc}")
    trained_eval = pd.DataFrame()

In [ ]:
def add_marginal_gain(df):
    if df.empty:
        return df
    out = []
    for lang, g in df.sort_values("vocab_size").groupby("language"):
        g = g.copy()
        # Lower tokens_per_char is better; equivalently higher chars_per_token is better.
        g["tokens_per_char"] = 1 / g["chars_per_token"]
        g["prev_tokens_per_char"] = g["tokens_per_char"].shift(1)
        g["relative_compression_gain"] = (g["prev_tokens_per_char"] - g["tokens_per_char"]) / g["prev_tokens_per_char"]
        out.append(g)
    return pd.concat(out, ignore_index=True)

def choose_vocab_size(df, min_gain=0.01):
    decisions = []
    if df.empty:
        return pd.DataFrame()
    for lang, g in add_marginal_gain(df).sort_values("vocab_size").groupby("language"):
        g = g.reset_index(drop=True)
        chosen = int(g.loc[len(g) - 1, "vocab_size"])
        reason = "largest tested size; gains did not fall below threshold"
        for i in range(1, len(g)):
            gain = g.loc[i, "relative_compression_gain"]
            if pd.notna(gain) and gain < min_gain:
                chosen = int(g.loc[i - 1, "vocab_size"])
                reason = f"marginal compression gain below {min_gain:.1%} after this point"
                break
        decisions.append({"language": lang, "chosen_vocab_size": chosen, "reason": reason})
    return pd.DataFrame(decisions)

if not trained_eval.empty:
    trained_eval_gain = add_marginal_gain(trained_eval)
    display(trained_eval_gain)
    display(choose_vocab_size(trained_eval, min_gain=0.01))

    plt.figure(figsize=(9, 5))
    for lang, g in trained_eval_gain.groupby("language"):
        plt.plot(g["vocab_size"], g["chars_per_token"], marker="o", label=lang)
    plt.xscale("log", base=2)
    plt.xlabel("Vocabulary size")
    plt.ylabel("Characters per token")
    plt.title("Tokenizer vocabulary size vs compression")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 8. How Many Tokens Are Needed?

Zipf's law alone does **not** determine how many tokens are needed to train a full language model. LM training data size depends on model size, compute budget, target quality, domain coverage, and evaluation goals. However, Zipf analysis can answer a narrower and useful question:

> How many observed tokens are enough to estimate a stable token-frequency distribution for this language/tokenizer?

The algorithm below subsamples increasing token counts and checks when the estimated Zipf exponent stabilizes.

In [ ]:
def alpha_stability(tokens, sample_sizes=None, repeats=5, seed=13):
    rng = np.random.default_rng(seed)
    n = len(tokens)
    if sample_sizes is None:
        sample_sizes = [1000, 3000, 10000, 30000, 100000, 300000, 1000000]
    sample_sizes = [s for s in sample_sizes if s <= n]
    rows = []
    arr = np.array(tokens, dtype=object)
    for size in sample_sizes:
        alphas = []
        vocab_observed = []
        for _ in range(repeats):
            if size == n:
                sample = arr
            else:
                idx = rng.choice(n, size=size, replace=False)
                sample = arr[idx]
            counter = Counter(sample.tolist())
            z = fit_zipf(counter)
            alphas.append(z["alpha"])
            vocab_observed.append(len(counter))
        rows.append({
            "sample_tokens": size,
            "alpha_mean": np.nanmean(alphas),
            "alpha_std": np.nanstd(alphas),
            "types_mean": np.mean(vocab_observed),
        })
    return pd.DataFrame(rows)

def recommend_min_tokens_for_zipf(stability_df, alpha_std_threshold=0.05):
    if stability_df.empty:
        return None
    stable = stability_df[stability_df["alpha_std"] <= alpha_std_threshold]
    if len(stable):
        return int(stable.iloc[0]["sample_tokens"])
    return int(stability_df.iloc[-1]["sample_tokens"])

stability_results = []
for lang, text in corpus.items():
    tokens = word_tokenize(text)
    stab = alpha_stability(tokens, repeats=5)
    stab["language"] = LANGUAGES[lang]
    stability_results.append(stab)

if stability_results:
    stability = pd.concat(stability_results, ignore_index=True)
    display(stability)
    recommendations = []
    for lang, g in stability.groupby("language"):
        recommendations.append({
            "language": lang,
            "minimum_tokens_for_stable_alpha_estimate": recommend_min_tokens_for_zipf(g),
        })
    display(pd.DataFrame(recommendations))

    plt.figure(figsize=(8, 5))
    for lang, g in stability.groupby("language"):
        plt.errorbar(g["sample_tokens"], g["alpha_mean"], yerr=g["alpha_std"], marker="o", capsize=3, label=lang)
    plt.xscale("log")
    plt.axhline(1.0, color="black", linestyle="--", linewidth=1, alpha=0.6)
    plt.xlabel("Sample tokens")
    plt.ylabel("Estimated Zipf alpha")
    plt.title("How many tokens are enough for a stable Zipf estimate?")
    plt.legend()
    plt.tight_layout()
    plt.show()

## 9. Final Answers

### Do tokens follow Zipf's law?

Usually yes, approximately. In the log-log rank-frequency plots, the middle ranks should look close to a straight declining line. The `alpha` and `r2` columns quantify this. The head and tail will deviate because of stop words, rare names, finite sample size, markup-like text, and tokenizer-specific subwords.

### What is different between Llama, Qwen, and Kimi tokenizers?

The difference is not just vocabulary size. The training corpus, algorithm, Unicode/byte handling, and multilingual coverage decide how much each language is fragmented. A tokenizer with more useful Hindi or Arabic pieces will show higher `chars_per_token`, fewer total tokens, and a less extreme rare-token tail on those languages. Kimi K2 reports a 160K vocabulary in its model card; Qwen tokenizers are also large multilingual/code tokenizers; Llama 3.1 is multilingual but supports a smaller official language set than a broad tokenizer-only vocabulary might imply.

### How does language matter?

Language changes the curve through morphology, script, whitespace conventions, and training-data representation. English generally receives strong coverage in popular LLM tokenizers. Hindi and Arabic can be split more if their scripts and morphemes were underrepresented during tokenizer training. Arabic clitics and Hindi compound/inflected forms can increase the long tail.

### Where is the sweet spot to stop tokenizer training?

Stop when increasing vocabulary size gives little additional compression on held-out text and starts mostly adding rare pieces. In this notebook the stopping rule uses marginal improvement in `tokens_per_char`. If the gain from doubling vocabulary falls below 1 percent, the previous vocabulary size is a good candidate. For real work, repeat this on a large held-out multilingual corpus and verify downstream LM validation loss.

### Can vocabulary size be decided algorithmically?

Yes, as a data-driven heuristic:

1. Train tokenizers at increasing vocab sizes.
2. Evaluate compression and Zipf fit per language on held-out data.
3. Reject sizes that hurt minority-language compression or create too many singleton tokens.
4. Choose the smallest vocabulary whose marginal compression gain is below a chosen threshold and whose downstream LM validation loss is not worse.

### How many tokens are needed to train a model that supports Zipf-like behavior?

For estimating Zipf behavior, use the stability algorithm above: increase sample size until `alpha` is stable across subsamples. For training an actual language model, Zipf is only a distributional diagnostic; it does not replace scaling-law decisions. You need enough tokens to cover the head, middle, and long tail of the target language and enough held-out evaluation to verify that the model learns them.

## References

- Zipf, G. K. (1949). *Human Behavior and the Principle of Least Effort*.
- Wikipedia API documentation: https://www.mediawiki.org/wiki/API:Main_page
- Hugging Face Transformers `AutoTokenizer`: https://huggingface.co/docs/transformers/main_classes/tokenizer
- Llama 3.1 model card/tokenizer repo: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B
- Qwen2.5 tokenizer repo: https://huggingface.co/Qwen/Qwen2.5-7B
- Kimi K2 model card/tokenizer repo: https://huggingface.co/moonshotai/Kimi-K2-Instruct